# Notebook 1 — MLflow LangChain Tracing Demo

This notebook walks through **MLflow's automatic tracing** for LangChain and LangGraph.

**What you'll learn:**
- How `mlflow.langchain.autolog()` captures every LLM call without code changes
- How to view traces in the MLflow UI
- How to manually add span context with `@mlflow.trace`
- How to compare traces across multiple runs

**Prerequisites:** Docker stack running (`docker compose up -d`)

In [ ]:
import sys
sys.path.insert(0, '../src')

import mlflow
from agents.config import configure_mlflow, get_ollama_base_url, get_ollama_model

# Point MLflow at our local tracking server
configure_mlflow(experiment_name='nb-01-tracing-demo')

# Enable automatic tracing — one line captures everything
mlflow.langchain.autolog()

print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Ollama model: {get_ollama_model()}')
print(f'Ollama host:  {get_ollama_base_url()}')

## 1. Simple LLM Call — Automatic Tracing

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

llm = ChatOllama(
    model=get_ollama_model(),
    base_url=get_ollama_base_url(),
)

with mlflow.start_run(run_name='single-llm-call'):
    response = llm.invoke([HumanMessage(content='What is MLflow in one sentence?')])
    print('Response:', response.content)

print('\n✓ Check the Traces tab in MLflow UI: http://localhost:5000')

## 2. LangChain Chain — Tracing Multiple Steps

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a concise technical explainer. Answer in 2-3 sentences.'),
    ('human', '{topic}'),
])

chain = prompt | llm | StrOutputParser()

topics = [
    'Explain MLflow experiment tracking',
    'Explain LangGraph StateGraph',
    'Explain Ollama local inference',
]

with mlflow.start_run(run_name='langchain-chain-trace'):
    for topic in topics:
        result = chain.invoke({'topic': topic})
        print(f'\n--- {topic} ---')
        print(result)

## 3. Manual Span Decoration with `@mlflow.trace`

In [ ]:
@mlflow.trace(name='custom_preprocessing', span_type='CHAIN')
def preprocess_topic(raw_topic: str) -> str:
    """Add context to a raw topic string before passing to LLM."""
    return f'In the context of Python MLOps: {raw_topic.strip().lower()}'

@mlflow.trace(name='postprocess_response', span_type='CHAIN')
def postprocess(text: str) -> dict:
    """Extract word count and clean response."""
    words = text.strip().split()
    return {'text': text.strip(), 'word_count': len(words)}

with mlflow.start_run(run_name='manual-span-trace'):
    topic = preprocess_topic('What are the benefits of model versioning?')
    raw_response = chain.invoke({'topic': topic})
    result = postprocess(raw_response)
    
    mlflow.log_metric('response_word_count', result['word_count'])
    print('Result:', result)

## 4. LangGraph Pipeline Tracing

In [ ]:
from agents.graph import pipeline

with mlflow.start_run(run_name='langgraph-trace'):
    mlflow.log_param('pipeline', 'researcher→writer→critic')
    
    final_state = pipeline.invoke({
        'topic': 'Why MLflow is essential for production ML teams',
        'messages': [],
        'research': '',
        'draft': '',
        'feedback': '',
        'final': '',
        'iterations': 0,
    })
    
    output = final_state.get('final') or final_state.get('draft', '')
    mlflow.log_text(output, 'output.md')
    mlflow.log_metric('iterations', final_state.get('iterations', 0))

print('Pipeline complete!')
print('\nFinal output (first 500 chars):')
print(output[:500])

## Summary

Open the MLflow UI at **http://localhost:5000** and explore:
- **Experiments** tab → `nb-01-tracing-demo` → click any run
- **Traces** tab → see every LLM call with latency, tokens, inputs/outputs
- **Artifacts** tab → download saved text outputs

Key takeaway: `mlflow.langchain.autolog()` instruments your entire LangChain stack with **zero additional code**.